In [1]:
import os
import sqlalchemy
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI

### Create the agent
An agent consists of a large language model (LLM) and a set of tools. Tools can simply be Python functions that the agent can call to perform specific tasks.

In [2]:
load_dotenv()

# initialize model using custom endpoint
llm = ChatOpenAI(
    model="gpt-4.1",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
)

In [3]:
# Replace with your actual connection details
engine = sqlalchemy.create_engine(
    "mssql+pyodbc://@atalsdbp01.aucklandtransport.govt.nz/ARTA_DW?driver=ODBC+Driver+17+for+SQL+Server&Trusted_Connection=yes"
)

db = SQLDatabase(
    engine,
    include_tables=[
        "fact_AIFS_Passenger_Journeys_New",
        "dim_route",
        "dim_mode",
        "dim_AIFS_Transaction_Type",
    ],
    indexes_in_table_info=True,
)
tools = SQLDatabaseToolkit(db=db, llm=llm).get_tools()

In [4]:
system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

Generally, all fact tables are best filtered using date_key columns, which are usually integers in the format YYYYMMDD.
For example, to filter for the month of January 15, 2020, you would use the condition date_key BETWEEN 20200101 AND 20200131.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
""".format(
    dialect=db.dialect,
    top_k=5,
)

In [5]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
)

### Run the agent
Running the agent involves sending it a user message and getting back a response. The response may include calls to tools, which the agent can use to get additional information before formulating its final response.

The response from the agent includes all messages exchanged during the conversation, including tool calls and their outputs.

In [6]:
question = "How many journeys were made in 2025?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

How many journeys were made in 2025?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_pdFYFYqAedPuPLW1tzXnTs0d)
 Call ID: call_pdFYFYqAedPuPLW1tzXnTs0d
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

dim_AIFS_Transaction_Type, dim_mode, dim_route, fact_AIFS_Passenger_Journeys_New
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_Q2kiDSg3BvY2JjM3yUS6tVZe)
 Call ID: call_Q2kiDSg3BvY2JjM3yUS6tVZe
  Args:
    table_names: fact_AIFS_Passenger_Journeys_New
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE [fact_AIFS_Passenger_Journeys_New] (
	[JourneyTransactionID] BIGINT NULL, 
	[TransactionID_TagOn] BIGINT NULL, 
	[TransactionID_TagOff] BIGINT N

## Snowflake connection
Try same as above but using Snowflake as the database backend.

In [8]:
# Set proxy if required
os.environ["HTTPS_PROXY"] = "http://at-proxy.aucklandtransport.govt.nz:8080"

# Create SQLAlchemy engine for Snowflake with externalbrowser authentication
engine = sqlalchemy.create_engine(
    f"snowflake://{os.getenv('SNOWFLAKE_USER')}@AUCKLANDTRANSPORT-DATA/DW_TST/DS_PRESENTATION"
    "?warehouse=WH_DATASCI_XS_TST&role=FR_DS_TST&authenticator=externalbrowser"
)

# Wrap with LangChain SQLDatabase
db = SQLDatabase(engine)

# Test: list tables
print(db.get_usable_table_names())

['dim_special_event', 'dim_weather_location', 'fact_calendar_features_daily', 'fact_macro_economic_features_daily', 'fact_special_event_features_daily', 'fact_weather_daily', 'fact_weather_historic_daily']


In [10]:
tools = SQLDatabaseToolkit(db=db, llm=llm).get_tools()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
)

In [11]:
question = "What's the average weather temperature in Auckland in 2023?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What's the average weather temperature in Auckland in 2023?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_L1uAHhS7ORGoCsBI2mviy2u4)
 Call ID: call_L1uAHhS7ORGoCsBI2mviy2u4
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

dim_special_event, dim_weather_location, fact_calendar_features_daily, fact_macro_economic_features_daily, fact_special_event_features_daily, fact_weather_daily, fact_weather_historic_daily
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_2F74eUVUkt0WaKYDQBe6n656)
 Call ID: call_2F74eUVUkt0WaKYDQBe6n656
  Args:
    table_names: fact_weather_daily,dim_weather_location
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE dim_w

In [12]:
question = "What's the historical average temperature for January?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What's the historical average temperature for January?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_jjniO1sWMM4BsNTpC4Qdaicz)
 Call ID: call_jjniO1sWMM4BsNTpC4Qdaicz
  Args:
    tool_input:
================================= Tool Message =================================
Name: sql_db_list_tables

dim_special_event, dim_weather_location, fact_calendar_features_daily, fact_macro_economic_features_daily, fact_special_event_features_daily, fact_weather_daily, fact_weather_historic_daily
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_uD0cGhqhc03ZJOTL3FxQb2ZW)
 Call ID: call_uD0cGhqhc03ZJOTL3FxQb2ZW
  Args:
    table_names: fact_weather_historic_daily
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE fact_w